# DistilBERT Sentiment Analysis

Fine-tunes DistilBERT on the IMDb reviews dataset for binary sentiment classification (3 epochs, Hugging Face `Trainer`).

## Fine-tune BERT for sentiment analysis on the IMDb dataset.

In [23]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset
import numpy as np

In [ ]:
imdb_dataset = load_dataset('stanfordnlp/imdb')
imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
imdb_dataset['train']['text'][1]

'"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn\'t matter what one\'s political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn\'t true. I\'ve seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don\'t exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies like The Brown Bunny, in which we\'re treated to the site of Vincent Gallo\'s throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) "double-standard" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious anatomical difference between men and women: there are no genitals on display when actresses appears nude, and the s

In [14]:
bert_tokenizer= AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
bert_distil_model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
def tokenize(batch):
    return bert_tokenizer(batch['text'], truncation=True, max_length=512)

dataset = imdb_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [17]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

In [26]:
data_collator = DataCollatorWithPadding(bert_tokenizer)

trraining_args = TrainingArguments(
    output_dir='bert_distil_ft',
    num_train_epochs = 3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    weight_decay=0.1,
    learning_rate=2e-5,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy='epoch',
    load_best_model_at_end=True,
)



In [27]:
def compute_metrics(eval_pred):
    logits,labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy" : (preds==labels).mean()}

In [30]:
trainer = Trainer(
    model = bert_distil_model,
    args = trraining_args,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
    processing_class=bert_tokenizer,
    data_collator = DataCollatorWithPadding(bert_tokenizer),
    compute_metrics=compute_metrics
)
trainer.train()
trainer.push_to_hub()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.945727,0.333117,0.921960
2,0.788803,0.271773,0.930120
3,0.592931,0.385059,0.930720


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/Cosminius/bert_distil_ft/commit/cc2e6db68439e13b179f03b32c14fc7e35cb2887', commit_message='End of training', commit_description='', oid='cc2e6db68439e13b179f03b32c14fc7e35cb2887', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Cosminius/bert_distil_ft', endpoint='https://huggingface.co', repo_type='model', repo_id='Cosminius/bert_distil_ft'), pr_revision=None, pr_num=None)